# NB02 — Market-Neutral Pairs Trading in Latin American ADRs

### Research Objective

This notebook develops and evaluates a market-neutral pairs trading framework for Latin American ADRs. The objective is to identify **stable, tradable pairs** by:

- Selecting candidate pairs **in-sample** using correlation, cointegration, and Ornstein–Uhlenbeck (OU) half-life filters.
- Locking hedge ratios (α, β) and OU model parameters on a fixed **training window**.
- Evaluating **out-of-sample** performance at both the individual pair level and within a multi-pair market-neutral portfolio, incorporating realistic transaction costs.

### Notebook Scope and Role in the Pipeline

This notebook implements a fully specified **rule-based Ornstein–Uhlenbeck (OU) pairs trading engine**, including cointegration screening, OU parameter estimation, and fixed entry/exit rules. The resulting trades and PnL series constitute a **frozen statistical baseline** that is consumed read-only by downstream execution and semantic risk-conditioning layers.

No macroeconomic data, news, sentiment, or regime classification is used at this stage.

## 1. Environment Setup

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP
#    - Imports
#    - Global CONFIG (single source of truth for all strategy parameters)
#    - Data directories
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint as eg_coint
from tqdm import tqdm

# --------------------------------------------------------------------------
# Plotting defaults
# --------------------------------------------------------------------------
plt.style.use("seaborn-v0_8")

# --------------------------------------------------------------------------
# Strategy CONFIG — every threshold used downstream lives here
# --------------------------------------------------------------------------
CONFIG = {
    # IS/OOS split
    "SPLIT_FRAC":   0.70,

    # Section 3.1 — Correlation prefilter
    "LOOKBACK":     756,    # IS lookback window (~3y trading days)
    "CORR_TH":      0.35,   # |ρ| threshold on IS daily returns

    # Section 3.2 — OU estimation & candidate filter
    "MIN_OBS":      252,    # min obs to fit OU on a pair
    "MIN_HL":       5,      # min half-life (days) — below this is noise
    "MAX_HL":       60,     # max half-life (days) — above is too slow for OU
    "MIN_R2":       0.05,   # min AR(1) R² for tradable OU dynamics

    # Section 4 — Pair selection & spread normalization
    "TOP_N":        20,     # number of OU pairs taken into the backtest
    "ROLL_WIN":     63,     # rolling window for OOS z-score (~3M)

    # Section 6 — Backtest rules
    "Z_ENTRY":      2.0,
    "Z_EXIT":       0.5,
    "TC_BPS":       5,      # round-trip transaction cost in basis points
    "ANNUAL_D":     252,
}

# --------------------------------------------------------------------------
# Data directories — project-root anchor (mirrors NB1)
# --------------------------------------------------------------------------
def find_project_root(markers=("pyproject.toml", ".git")):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if any((cand / m).exists() for m in markers):
            return cand
    return here  # fallback: current working dir

ROOT      = find_project_root()
DATA_DIR  = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts" / "nb2_outputs"

FILES = {
    "universe": DATA_DIR / "tickers_pairs.csv",
    "prices":   DATA_DIR / "prices_pairs.csv",
    "returns":  DATA_DIR / "returns_pairs.csv",
}

print(f"[INFO] Project root: {ROOT}")

## 2. Load & Prepare Pairs Universe
This section loads the filtered ADR universe produced in NB01, aligns price and return series, and applies a fixed train/test split. The diagnostics below verify data integrity and ensure consistency between the in-sample estimation window and the out-of-sample evaluation period.

**Regime context.** The 70/30 split places the OOS window from mid-2022 onward, which encompasses the Fed tightening cycle and post-pandemic structural reset. Academic research documents a secular decline in simple pairs-trading profitability predating this window (Gatev et al., 2006; Do & Faff, 2010); the specific mid-2022 / Fed-tightening characterization of the current OOS window is the notebook's own framing of that deterioration extending into the recent period. The OOS evaluation should therefore be read as a stress test of IS-fitted parameters through a known adverse regime, not as performance under neutral conditions.

In [ ]:
## 2. Load & Prepare Pairs Universe
# ==============================================================================
# 2. DATA LOADING & PRE-PROCESSING (Pairs Universe)
# ==============================================================================

# === 1) Load Universe ===
universe_df = pd.read_csv(FILES["universe"])
tick_col = "Ticker" if "Ticker" in universe_df.columns else universe_df.columns[0]
universe = sorted(set(universe_df[tick_col].astype(str)))

# === 2) Load Price & Return Panels ===
price = pd.read_csv(FILES["prices"],  parse_dates=["Date"], index_col="Date").sort_index()
retd  = pd.read_csv(FILES["returns"], parse_dates=["Date"], index_col="Date").sort_index()

# Normalize tickers (strip "_Price")
price.columns = [c.replace("_Price", "") for c in price.columns]
retd.columns  = [c.replace("_Price", "") for c in retd.columns]

# === 3) Enforce ticker intersection ===
common  = sorted(set(universe) & set(price.columns) & set(retd.columns))
missing = sorted(set(universe) - set(common))

if missing:
    print("[WARN] Dropping tickers missing from price/returns:", missing)

universe = common
price    = price[universe]
retd     = retd[universe]

print(f"[OK] Universe loaded: {len(universe)} tickers")
print(f"[OK] Prices:  {price.shape}  (time × tickers)")
print(f"[OK] Returns: {retd.shape}   (time × tickers)")

# IS/OOS Split (configurable via CONFIG['SPLIT_FRAC'])

split_idx  = int(CONFIG["SPLIT_FRAC"] * len(price))
split_date = price.index[split_idx]

price_train, price_test = price.iloc[:split_idx], price.iloc[split_idx:]
retd_train,  retd_test  = retd.iloc[:split_idx],  retd.iloc[split_idx:]

assert price_train.index.intersection(price_test.index).empty
assert retd_train.index.intersection(retd_test.index).empty
assert len(price_train) + len(price_test) == len(price)
assert len(retd_train) + len(retd_test) == len(retd)

print(f"[OK] Train/Test shapes: {price_train.shape} / {price_test.shape}")
print(f"[OK] Split date: {split_date.date()}")

## Ornstein–Uhlenbeck (OU) Mean-Reversion Framework  
*A pragmatic alternative to traditional cointegration-based approaches.*


OU models target **empirical mean reversion**, yielding stable, tradeable parameters (half-life, $\phi$, $R^2$, volatility) without requiring strict stationarity. The approach of modeling the spread between a cointegrated pair as an Ornstein–Uhlenbeck process was formalized by Elliott, van der Hoek & Malcolm (2005).

---

### OU Dynamics
Continuous-time:
$$ dS_t=\kappa(\mu-S_t)\,dt+\sigma\,dW_t,\qquad \kappa>0. $$

Discrete AR(1):
$$ S_t=\phi S_{t-1}+\varepsilon_t,\qquad \phi=e^{-\kappa\Delta t}. $$

Estimated form:
$$ S_t = a + \phi S_{t-1} + \varepsilon_t. $$

Half-life:
$$ \text{HL}=-\frac{\ln 2}{\ln \phi}. $$

Viability:
$$ 0<\phi<1,\quad 5\le\text{HL}\le60,\quad R^2\ge0.05. $$

---

### Spread Construction
Hedge ratio:
$$ Y_t=\alpha+\beta X_t+u_t. $$

Spread:
$$ S_t=Y_t-(\alpha+\beta X_t). $$

---

### Why OU Works
- Robust to short estimation windows where strict I(1) testing fails  
- Cointegration tests (Engle-Granger, ADF on the spread) are computed and reported as diagnostics in the candidate dataframe, but final selection is driven by OU criteria (φ, half-life, R²) rather than cointegration p-value cutoffs. This deviates from the strict 95% / 90% confidence thresholds in the institutional framework and reflects a deliberate trade-off given the smaller LatAm universe (69 names).
- Intuitive parameters  
- Produces more viable pairs in practice  

---

### Workflow
1. Correlation screen  
2. Estimate $(\alpha,\beta)$  
3. Build $S_t$  
4. Fit AR(1) → $\phi$, HL, $R^2$  
5. Keep pairs with strong mean reversion  

---

### Interpretation
- Small HL → fast convergence  
- Large HL → slow decay  
- High $R^2$ → stable dynamics  

OU parameters are estimated on the in-sample window and treated as fixed during out-of-sample evaluation and trade simulation.

## 3. OU Candidate Selection Pipeline
*Correlation Screening → Cointegration Tests → OU Estimation*

### 3.1 In-Sample (IS) Window & Correlation Prefilter

An initial correlation prefilter is applied on the in-sample window to restrict the candidate set to pairs with sufficient co-movement, reducing estimation noise and computational burden.

**Threshold choice.** The 0.35 threshold is applied to **return** correlations, which is the institutionally-standard measure of comovement and is robust to common trends. This is intentionally looser than the 0.6 threshold commonly used by practitioners for **price** correlations — a convention rooted in the distance/price-level pairs-selection tradition established by Gatev et al. (2006): price-level correlations are contaminated by shared drift (two unrelated assets that both trend upward can show ρ_price > 0.9 with no real economic relationship), whereas returns correlations isolate genuine comovement. The lower threshold also compensates for the smaller LatAm universe (69 names).

**Role in the pipeline.** This step functions as a computational prefilter to avoid running OU estimation on all C(69,2) = 2,346 pairs. The binding selection criterion is the OU half-life and R² filter that follows in Section 3.2 — that filter rejects roughly 60% of prefilter-passing pairs, doing the load-bearing selection work.

In [ ]:
## 3.1 In-Sample Window & Correlation Prefilter
# ======================================================================

# Parameters
LOOKBACK = CONFIG["LOOKBACK"]
CORR_TH  = CONFIG["CORR_TH"]

# Slice IS lookback
price_lb = price_train.iloc[-LOOKBACK:].dropna(how="all")
retd_lb  = retd_train.iloc[-LOOKBACK:].dropna(how="all")

# Correlation screen (IS only)
corr = retd_lb.corr()
pairs_prefilter = [
    (u1, u2, corr.loc[u1, u2])
    for i, u1 in enumerate(universe)
    for u2 in universe[i+1:]
    if abs(corr.loc[u1, u2]) >= CORR_TH
]

print(f"[INFO] Prefilter: {len(pairs_prefilter)} pairs |ρ| ≥ {CORR_TH}")

### 3.2 Hedge Ratio (α, β), EG/ADF, OU Fit & Final Filter

In [ ]:
## 3.2 Hedge Ratio (α, β), EG/ADF Diagnostics, AR(1) OU Fit
MIN_OBS = CONFIG["MIN_OBS"]
MIN_HL  = CONFIG["MIN_HL"]
MAX_HL  = CONFIG["MAX_HL"]
MIN_R2  = CONFIG["MIN_R2"]
rows = []

for t1, t2, rho in tqdm(pairs_prefilter, desc="OU Estimation", ncols=100):

    df = price_lb[[t1, t2]].dropna()
    if len(df) < MIN_OBS: 
        continue

    y, x = df[t1], df[t2]

    # α + β hedge ratio
    beta, alpha = np.polyfit(x, y, 1)
    S = (y - (alpha + beta * x)).dropna()
    if len(S) < MIN_OBS:
        continue

    # EG / ADF diagnostics
    try:    eg_p = eg_coint(y, x)[1]
    except: eg_p = np.nan
    try:    adf_p = adfuller(S)[1]
    except: adf_p = np.nan

    # OU via AR(1)
    lag = S.shift(1).dropna()
    ou  = sm.OLS(S.loc[lag.index], sm.add_constant(lag)).fit()

    phi = float(ou.params.iloc[1])       # FIXED: slope extraction
    hl  = -np.log(2) / np.log(phi) if 0 < phi < 1 else np.inf

    rows.append({
        "Ticker1": t1, "Ticker2": t2, "rho": rho,
        "alpha": alpha, "beta": beta,
        "phi": phi, "halflife": hl,
        "r2": float(ou.rsquared),
        "eg_p": eg_p, "adf_p": adf_p,
        "Nobs": len(S)
    })

df_ou = pd.DataFrame(rows)
# Strategy claim is "market-neutral pairs trading" — this requires β > 0
# so that the constructed spread offsets long and short legs. β < 0
# would make both legs enter on the same side of the position
# (long-long or short-short), violating market neutrality. EG
# regressions occasionally produce β < 0 for weakly-correlated or
# non-cointegrated pairs; we exclude them at the candidate-filter
# level rather than at top-N selection so the diagnostics in
# df_candidates reflect only economically valid pairs.
df_candidates = df_ou[
    df_ou["phi"].between(0, 1) &
    df_ou["halflife"].between(MIN_HL, MAX_HL) &
    (df_ou["r2"] >= MIN_R2) &
    (df_ou["beta"] > 0)
].sort_values("halflife")

print(f"[INFO] OU estimated for {len(df_ou)} pairs.")
print(f"[INFO] Tradable OU candidates: {len(df_candidates)}")
df_candidates.head(20)

*All statistics reported above are computed strictly in-sample and are used solely for candidate selection and parameter locking* 

## 4. Spread Construction, Normalization & Signal Engineering

**Objective.** Convert OU-selected pairs into tradable mean-reversion signals using hedge-ratio regression, spread construction, and rolling statistical normalization.
In NB02, the hedge ratios $(\alpha, \beta)$ are estimated **in-sample** and then applied **out-of-sample**.

While the OU framework admits a closed-form stationary distribution and an associated equilibrium S-score, this notebook employs a rolling z-score normalization for signal construction. This choice trades off model fidelity for adaptiveness: the OU S-score uses IS-frozen κ and σ_eq, which can go stale if the spread's volatility regime shifts in the OOS period; a rolling z-score adapts to OOS volatility drift at the cost of treating the spread as a generic stationary process rather than as an OU realization. Refinements that re-introduce the OU S-score with rolling parameter updates are deferred to downstream layers.

---

### 1. Hedge-ratio estimation (train window)

We regress $Y_t$ on $X_t$ to obtain the static parameters:
$$
Y_t = \alpha + \beta X_t + u_t.
$$

---

### 2. Spread construction (test window, using frozen parameters)

Out-of-sample spreads are built as:
$$
S_t = Y_t - \left(\alpha + \beta X_t\right).
$$

---

### 3. Rolling z-score normalization

For a rolling window of size $w$:
$$
Z_t = \frac{S_t - \mu_t}{\sigma_t},
$$

where $\mu_t$ and $\sigma_t$ denote the rolling mean and rolling standard deviation of the spread.

---

### 4. Signal rules

* **Long-spread entry:**
  $$
  Z_t < -z_{\text{entry}}
  $$

* **Short-spread entry:**
  $$
  Z_t > z_{\text{entry}}
  $$

* **Exit to flat:**
  $$
  \lvert Z_t \rvert < z_{\text{exit}}
  $$

These standardized signals form the basis of our market-neutral trading logic in the out-of-sample evaluation.

### 4.1 Select Top OU Pairs
To construct a manageable and interpretable trading universe, we retain a fixed subset of OU-qualified pairs ranked by fastest estimated half-life. This selection criterion is based solely on in-sample mean-reversion speed and does not incorporate any out-of-sample performance information.

In [ ]:
## 4.1. Select Top OU Pairs
TOP_N    = CONFIG["TOP_N"]
ROLL_WIN = CONFIG["ROLL_WIN"]

df_top = (
    df_candidates
    .sort_values("halflife")
    .head(TOP_N)
    .assign(pair=lambda d: d["Ticker1"] + "-" + d["Ticker2"])
    .reset_index(drop=True)
)

print(f"[INFO] Using Top {TOP_N} OU pairs (fastest half-life):")
display(df_top[["pair","Ticker1","Ticker2","rho","phi","halflife","r2"]].head(10))

### 4.2 Spread & Rolling Z-Scores
For each selected pair, out-of-sample hedge-adjusted spreads and rolling z-scores are constructed using parameters estimated exclusively on the in-sample window.

In [ ]:
## 4.2 OOS Spread & Z-Score Construction (NB2, compact)
# Spreads are built on the FULL OOS index (no dropna): missing-leg days are
# kept as NaN so they can be handled explicitly downstream (zero PnL, no
# gap-spanning diff) instead of being silently compressed out of the series.
pair_panels = {}
zscore_wide = pd.DataFrame(index=price_test.index)

for _, r in df_top.iterrows():
    t1, t2 = r["Ticker1"], r["Ticker2"]
    alpha, beta = r["alpha"], r["beta"]
    name = f"{t1}-{t2}"

    # OOS spread using frozen IS alpha, beta — kept on the full OOS index
    s = price_test[t1] - (alpha + beta * price_test[t2])

    # Rolling z-score; require a full window so warm-up / post-gap bars stay NaN
    roll = s.rolling(ROLL_WIN, min_periods=ROLL_WIN)
    z = (s - roll.mean()) / roll.std()
    z = z.replace([np.inf, -np.inf], np.nan)

    panel = pd.DataFrame({"spread": s, "zscore": z})
    pair_panels[name] = panel
    zscore_wide[name] = z

print(f"[INFO] Built OOS spread/z-score panels: {len(pair_panels)}")
print(f"[INFO] Z-score matrix shape: {zscore_wide.shape}")

### 4.3 Diagnostic Visualization
The following diagnostic illustrates the hedge-adjusted price relationship and the corresponding out-of-sample z-score dynamics for a representative pair.

In [ ]:
## 4.3 Diagnostic Plot for Fastest OU Pair
ex = df_top.iloc[0]
t1, t2, alpha, beta = ex["Ticker1"], ex["Ticker2"], ex["alpha"], ex["beta"]
name = f"{t1}-{t2}"
panel = pair_panels[name]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12,7), sharex=True)

# Price legs
ax1.plot(price_test[t1], label=t1)
ax1.plot(alpha + beta * price_test[t2], label=f"{t2} × β + α")
ax1.set_title(f"{name}: OOS Hedge Relationship")
ax1.set_ylabel("Price"); ax1.legend()

# Z-Score
ax2.plot(panel["zscore"], label="Z-score")
for h in [0, 2, -2]:
    ax2.axhline(h, linestyle="--", linewidth=1, color=("gray" if h==0 else "red"))
ax2.set_title(f"{name}: {ROLL_WIN}-day Rolling Z-Score (OOS)")
ax2.set_ylabel("Z-score"); ax2.legend()

plt.tight_layout()
plt.show()

## 5. Backtest-Ready Data Assembly (Top-N Spreads & Z-Scores)
This section assembles the final data structures required for systematic backtesting. All spreads and z-scores are constructed out-of-sample using parameters estimated on the training window. No trading decisions, portfolio allocation, or performance evaluation are performed at this stage.

In [ ]:
# ============================================================================
# 5. Backtest-Ready Data Assembly (Top-N Spreads & Z-Scores)
# ============================================================================

pairs_top = df_top.copy()

# Wide matrices (OOS index only)
S_pairs = pd.DataFrame({name: p["spread"] for name, p in pair_panels.items()})
Z_pairs = zscore_wide[pair_panels.keys()]  # keep column order consistent

# Optional cleanup: drop columns that are fully NaN (rare but safe)
S_pairs = S_pairs.dropna(axis=1, how="all")
Z_pairs = Z_pairs.dropna(axis=1, how="all")

print("[INFO] Backtest variables constructed:")
print("pairs_top:", pairs_top.shape)
print("S_pairs:   ", S_pairs.shape)
print("Z_pairs:   ", Z_pairs.shape)

## 6. Per-Pair OU Z-Score Backtest (Out-of-Sample)
This evaluates the statistical behavior of individual OU pairs — not a deployable strategy

### 6.1 Backtest Initialization & OOS Alignment
This section initializes the per-pair backtesting logic and aligns all signals, positions, and PnL strictly to the out-of-sample evaluation window.

In [ ]:
# ============================================================================
# 6.1 Preconditions, OOS Alignment, Parameters
# ============================================================================

assert "pairs_top" in globals(), "Run Top-N selection first."
assert "S_pairs"   in globals(), "Run spread/z-score construction first."
assert "Z_pairs"   in globals(), "Run spread/z-score construction first."
assert "price_test" in globals(), "Need OOS price panel."

# Restrict strictly to the test (OOS) window
idx_oos = price_test.index
S_pairs = S_pairs.reindex(idx_oos).sort_index()
Z_pairs = Z_pairs.reindex(idx_oos)

# Backtest parameters — sourced from CONFIG (single source of truth)
Z_ENTRY  = CONFIG["Z_ENTRY"]
Z_EXIT   = CONFIG["Z_EXIT"]
TC_BPS   = CONFIG["TC_BPS"]
ANNUAL_D = CONFIG["ANNUAL_D"]

print("[INFO] Backtest engine initialized.")
print("OOS rows:", len(idx_oos))

### 6.2 Per-Pair Trading Logic (State Machine & PnL Accounting)

Trading rules are applied independently to each pair using fixed entry
and exit thresholds, without portfolio-level coordination or dynamic
risk adjustment.

**State machine.** At each bar, the position state is updated as follows:
- If $|Z_t| < z_{\text{exit}}$ → flat (state = 0)
- Else if $Z_t \le -z_{\text{entry}}$ → long spread (state = +1)
- Else if $Z_t \ge +z_{\text{entry}}$ → short spread (state = −1)
- Otherwise → state unchanged (hold previous position)

**Direct flips are allowed.** If $Z_t$ moves from below $-z_{\text{entry}}$
to above $+z_{\text{entry}}$ in a single step (without passing through the
dead zone $|Z_t| < z_{\text{exit}}$), the position flips directly from
long to short. This captures both legs of a violent reversal at the cost
of an additional transaction-cost payment for the implicit close-and-reopen.
The alternative — requiring a flat reset between long and short trades —
is more conservative but misses the second leg of true reversals.

**NaN handling.** If $Z_t$ is undefined (rolling-window warm-up, data
gaps), no signal fires and the position state persists from the prior
bar. This matches how a real trader handles missing data: hold, don't
exit on a gap.

**PnL accounting.** PnL is computed as $\text{pos}_{t-1} \cdot \Delta S_t$
(lagged position to avoid look-ahead), in dollars-per-unit-of-spread.
Transaction costs are levied on $|\Delta\text{pos}|$ at $\text{TC}_{\text{bps}}$
bps of total notional ($|Y_t| + |\beta X_t|$). See unit caveat in Section 6.3.

In [ ]:
# ============================================================================
# 6.2 Backtest Class + Backtest Function
# ============================================================================

@dataclass
class PairBacktestResult:
    pair: str; t1: str; t2: str; beta: float
    ann_pnl: float; ann_vol: float; sharpe: float; max_dd: float; n_trades: int
    pnl: pd.Series; pos: pd.Series; z: pd.Series



def backtest_pair(row, S_pairs, Z_pairs,
                  z_entry=Z_ENTRY, z_exit=Z_EXIT, tc_bps=TC_BPS):

    name = row["pair"]
    t1, t2, beta = row["Ticker1"], row["Ticker2"], float(row["beta"])

    # Work on the FULL OOS index; NaN (warm-up / data gaps) is preserved so it
    # can be handled explicitly rather than compressed away by dropna(). Every
    # pair therefore spans the full OOS window -> clean cross-pair alignment.
    s = S_pairs[name].reindex(price_test.index)
    z = Z_pairs[name].reindex(price_test.index)   # NaN preserved (no fillna)

    # --- Signals (NaN-safe: NaN comparisons return False, so no signal fires) ---
    long_sig  = z <= -z_entry
    short_sig = z >=  z_entry
    flat_sig  = z.abs() < z_exit

    # --- Position state machine ---
    # Direct flips (+1 <-> -1) allowed when |z| crosses +/-z_entry without passing
    # through the |z| < z_exit dead zone. NaN z (warm-up, data gaps) -> no signal
    # change, state persists from the previous bar.
    pos = pd.Series(0.0, index=z.index)
    state = 0.0
    for i, (L, S_, F) in enumerate(zip(long_sig, short_sig, flat_sig)):
        if F:    state = 0.0
        elif L:  state = 1.0
        elif S_: state = -1.0
        # else: state unchanged (NaN, or |z| in [z_exit, z_entry] dead zone)
        pos.iloc[i] = state

    # --- PnL ---
    # Mark-to-market only across consecutive OBSERVED bars. Where the current or
    # prior spread is missing (gap), dS is forced to 0 so a multi-day hole is
    # never booked as a single-bar jump.
    dS    = s.diff()
    valid = s.notna() & s.shift(1).notna()
    dS    = dS.where(valid, 0.0)
    gross = pos.shift(1).fillna(0.0) * dS

    # Notional for TC (missing-leg days contribute no notional)
    y_px = price_test[t1].reindex(price_test.index).abs()
    x_px = (beta * price_test[t2].reindex(price_test.index)).abs()
    notional = (y_px + x_px).fillna(0.0)

    trades = pos.diff().abs().fillna(0.0)
    tc = notional * trades * (tc_bps * 1e-4)

    pnl = (gross - tc).fillna(0.0)   # full OOS index, no NaN

    # --- Performance ---
    ann_pnl = pnl.mean() * ANNUAL_D
    ann_vol = pnl.std(ddof=0) * np.sqrt(ANNUAL_D)
    sharpe  = ann_pnl / ann_vol if ann_vol > 0 else np.nan

    equity = pnl.cumsum()
    max_dd = (equity - equity.cummax()).min()
    n_trades = int((trades > 0).sum())

    return PairBacktestResult(name, t1, t2, beta, ann_pnl, ann_vol, sharpe,
                              max_dd, n_trades, pnl, pos, z)

### 6.3 Batch Evaluation Across OU Candidates
The table below summarizes out-of-sample performance statistics for each OU-qualified pair traded independently under identical rule specifications.

In [ ]:
# ============================================================================
# 6.3 Run Backtests for All Top-N Pairs
# ============================================================================

results = [backtest_pair(r, S_pairs, Z_pairs) for _, r in pairs_top.iterrows()]

bt_summary = pd.DataFrame({
    "pair":    [r.pair for r in results],
    "Ticker1": [r.t1 for r in results],
    "Ticker2": [r.t2 for r in results],
    "beta":    [r.beta for r in results],
    "AnnPnL":  [r.ann_pnl for r in results],
    "AnnVol":  [r.ann_vol for r in results],
    "Sharpe":  [r.sharpe for r in results],
    "MaxDD":   [r.max_dd for r in results],
    "Trades":  [r.n_trades for r in results],
}).sort_values("Sharpe", ascending=False)

print("[INFO] Backtests completed (OOS only).")

# Merge eg_p from df_top for visibility
bt_display = bt_summary.merge(
    df_top[["pair", "eg_p", "halflife"]], on="pair", how="left"
)
cols = ["pair", "Ticker1", "Ticker2", "beta", "halflife", "eg_p",
        "AnnPnL", "AnnVol", "Sharpe", "MaxDD", "Trades"]
display(bt_display[cols].head(10).round(4))

> **Unit caveat.** AnnPnL and MaxDD are in spread-dollar units (per 1-unit
> of position, where "1 unit" = long 1 share of leg 1 + short β shares of
> leg 2). They are NOT percentage returns — pairs with high-priced legs
> (e.g., BAP-IFS, β=5.4) have larger natural PnL magnitudes than pairs
> with low-priced legs. Sharpe is unit-invariant and remains directly
> comparable across pairs. These metrics are reported for comparative
> diagnostics across pairs and should not be interpreted as indicative of
> deployable portfolio performance.

### 6.4 Visual Diagnostics (Equity Curve, Positioning, Z-Score)
The following plots are provided for qualitative validation of signal behavior, position dynamics, and mean-reversion consistency at the individual pair level.

In [ ]:
# ============================================================================
# 6.4 Diagnostic Plot Function
# ============================================================================

def plot_pair_diagnostics(result: PairBacktestResult, title_suffix=""):
    fig, (ax0, ax1, ax2) = plt.subplots(3, 1, figsize=(12,8), sharex=True)

    ax0.plot(result.pnl.cumsum(), label="PnL")
    ax0.set_title(f"{result.pair}: Equity Curve {title_suffix}")
    ax0.set_ylabel("PnL")

    ax1.step(result.pos.index, result.pos, where="post")
    ax1.set_title("Position")
    ax1.set_ylabel("Units")

    ax2.plot(result.z, label="Z-score")
    ax2.axhline(Z_ENTRY,  ls="--", c="red")
    ax2.axhline(-Z_ENTRY, ls="--", c="red")
    ax2.axhline(Z_EXIT,   ls=":",  c="grey")
    ax2.axhline(-Z_EXIT,  ls=":",  c="grey")
    ax2.set_title("Z-Score")
    ax2.set_ylabel("Z")

    plt.tight_layout()
    plt.show()

### 6.5 Case Study: Representative High-Sharpe OU Pairs (OOS)
Selected examples are shown for illustration of typical out-of-sample dynamics and are not intended to represent optimized or preferred trading candidates.

In [ ]:
# ============================================================================
# 6.5 Plot Top (x) Sharpe Pairs
# ============================================================================

top_pairs = bt_summary.head(1)["pair"].tolist()

for p in top_pairs:
    res = next(r for r in results if r.pair == p)
    plot_pair_diagnostics(res, title_suffix="(OOS)")

## 7. Cointegration-Only Baseline

To make the contribution of the OU candidate filter empirically measurable rather than literature-cited, this section reruns the same pipeline on a pair set selected by **cointegration alone** — dropping the AR(1) OU filters on half-life, R², and φ.

What stays constant: the correlation prefilter (|ρ| ≥ 0.35), the Engle-Granger cointegration screen, the hedge-ratio estimation (α, β via OLS), the rolling z-score construction (`ROLL_WIN = 63`), the entry/exit thresholds (`Z_ENTRY = 2.0`, `Z_EXIT = 0.5`), and the `backtest_pair()` engine itself.

What changes: instead of filtering by OU dynamics (φ, half-life, R²) and selecting top-N by *fastest half-life*, we keep all cointegrated pairs (`eg_p < 0.05`, `β > 0`) and select top-N by *smallest* EG cointegration p-value (most cointegrated).

Artifacts are exported with a `_coint` suffix so NB04 can ingest both pair sets side by side and build a 4-variant ablation: `Cointegration-only · OU · OU+earnings · OU+earnings+sentiment`.

### 7.1 Candidate filter, Top-N selection, OOS spread/z-score

In [ ]:
# ============================================================================
# 7.1 Cointegration-only candidate filter + Top-N selection
# ----------------------------------------------------------------------------
# Drop the OU filters (phi, halflife, r2). Keep:
#   • Engle-Granger cointegration: eg_p < COINT_P_TH
#   • Market-neutral constraint:   beta > 0
# Top-N by smallest EG p-value (most cointegrated), matching TOP_N from CONFIG.
# ============================================================================

COINT_P_TH = 0.05  # standard EG significance threshold

df_candidates_coint = df_ou[
    (df_ou["eg_p"] < COINT_P_TH) &
    (df_ou["beta"] > 0)
].sort_values("eg_p").reset_index(drop=True)

print(f"[INFO] Cointegration-only candidates: {len(df_candidates_coint)}")
print(f"[INFO] (vs OU-filtered candidates:    {len(df_candidates)})")

df_top_coint = (
    df_candidates_coint
    .head(TOP_N)
    .assign(pair=lambda d: d["Ticker1"] + "-" + d["Ticker2"])
    .reset_index(drop=True)
)

print(f"\n[INFO] Top {TOP_N} cointegration-only pairs (smallest EG p-value):")
display(df_top_coint[["pair","Ticker1","Ticker2","rho","eg_p","beta","halflife"]].head(10).round(4))

# ----------------------------------------------------------------------------
# OOS spread + rolling z-score (same construction as §4.2, applied to df_top_coint)
# ----------------------------------------------------------------------------
pair_panels_coint = {}
zscore_wide_coint = pd.DataFrame(index=price_test.index)

for _, r in df_top_coint.iterrows():
    t1, t2 = r["Ticker1"], r["Ticker2"]
    alpha, beta = r["alpha"], r["beta"]
    name = f"{t1}-{t2}"

    s = price_test[t1] - (alpha + beta * price_test[t2])
    roll = s.rolling(ROLL_WIN, min_periods=ROLL_WIN)
    z = (s - roll.mean()) / roll.std()
    z = z.replace([np.inf, -np.inf], np.nan)

    pair_panels_coint[name] = pd.DataFrame({"spread": s, "zscore": z})
    zscore_wide_coint[name] = z

pairs_top_coint = df_top_coint.copy()
S_pairs_coint   = pd.DataFrame({n: p["spread"] for n, p in pair_panels_coint.items()})
Z_pairs_coint   = zscore_wide_coint[pair_panels_coint.keys()]

S_pairs_coint = S_pairs_coint.dropna(axis=1, how="all")
Z_pairs_coint = Z_pairs_coint.dropna(axis=1, how="all")

print(f"\n[INFO] Baseline backtest inputs assembled:")
print(f"  pairs_top_coint: {pairs_top_coint.shape}")
print(f"  S_pairs_coint:   {S_pairs_coint.shape}")
print(f"  Z_pairs_coint:   {Z_pairs_coint.shape}")

### 7.2 Run baseline backtest (same engine as §6)

In [ ]:
# ============================================================================
# 7.2 Backtest the baseline pair set with the existing backtest_pair() engine
# ============================================================================
results_coint = [
    backtest_pair(r, S_pairs_coint, Z_pairs_coint)
    for _, r in pairs_top_coint.iterrows()
]

bt_summary_coint = pd.DataFrame({
    "pair":    [r.pair    for r in results_coint],
    "Ticker1": [r.t1      for r in results_coint],
    "Ticker2": [r.t2      for r in results_coint],
    "beta":    [r.beta    for r in results_coint],
    "AnnPnL":  [r.ann_pnl for r in results_coint],
    "AnnVol":  [r.ann_vol for r in results_coint],
    "Sharpe":  [r.sharpe  for r in results_coint],
    "MaxDD":   [r.max_dd  for r in results_coint],
    "Trades":  [r.n_trades for r in results_coint],
}).sort_values("Sharpe", ascending=False)

print("[INFO] Cointegration-only baseline backtests completed (OOS only).")

bt_display_coint = bt_summary_coint.merge(
    pairs_top_coint[["pair", "eg_p", "halflife"]], on="pair", how="left"
)
cols = ["pair", "Ticker1", "Ticker2", "beta", "halflife", "eg_p",
        "AnnPnL", "AnnVol", "Sharpe", "MaxDD", "Trades"]
display(bt_display_coint[cols].head(10).round(4))

### 7.3 Export baseline artifacts (parallel to NB2 → NB3 export, `_coint` suffix)

In [ ]:
# ============================================================================
# 7.3 Export baseline artifacts (mirrors NB2 → NB3 schema, _coint suffix)
# ============================================================================

# Define OUT_DIR locally (the existing NB2 → NB3 export cell does this too,
# but it runs after §7 in cell order; mirroring it here keeps §7 self-contained).
OUT_DIR = ARTIFACTS
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Pair metadata ---
pair_meta_coint = pairs_top_coint[
    ["pair", "Ticker1", "Ticker2", "rho", "alpha", "beta",
     "phi", "halflife", "r2", "eg_p", "adf_p", "Nobs"]
].copy()
pair_meta_coint.to_csv(OUT_DIR / "pairs_metadata_coint.csv", index=False)
print("[OK] pairs_metadata_coint.csv saved:", pair_meta_coint.shape)

# --- Pair-level time series (long format, full OOS index) ---
rows_coint = []
oos_idx_coint = price_test.index
for r in results_coint:
    df = pd.DataFrame({
        "date":     oos_idx_coint,
        "pair":     r.pair,
        "spread":   S_pairs_coint[r.pair].reindex(oos_idx_coint),
        "zscore":   r.z.reindex(oos_idx_coint),
        "position": r.pos.reindex(oos_idx_coint),
        "pnl":      r.pnl.reindex(oos_idx_coint).fillna(0.0),
    })
    rows_coint.append(df)

pair_ts_coint = (
    pd.concat(rows_coint, axis=0)
      .sort_values(["pair", "date"])
      .reset_index(drop=True)
)

expected_rows_coint = len(price_test) * len(pairs_top_coint)
assert len(pair_ts_coint) == expected_rows_coint, (
    f"pair_ts_coint row count mismatch: got {len(pair_ts_coint)}, "
    f"expected {expected_rows_coint}"
)
pair_ts_coint.to_parquet(OUT_DIR / "pair_timeseries_coint.parquet", index=False)
print("[OK] pair_timeseries_coint.parquet saved:", pair_ts_coint.shape)

# --- Trade-level table (event-based, direct-flip aware, mirrors cell-35 logic) ---
trade_rows_coint = []
for r in results_coint:
    pos = r.pos
    z   = r.z
    pnl = r.pnl

    active     = False
    entry_date = None
    entry_z    = None
    entry_pos  = None
    direction  = None
    trade_pnl  = 0.0
    prev_p     = 0.0

    for t in pos.index:
        p = pos.loc[t]
        opened  = (prev_p == 0) and (p != 0)
        closed  = (prev_p != 0) and (p == 0)
        flipped = (prev_p != 0) and (p != 0) and (np.sign(p) != np.sign(prev_p))

        if active and (closed or flipped):
            trade_rows_coint.append({
                "pair":       r.pair,
                "entry_date": entry_date,
                "exit_date":  t,
                "direction":  direction,
                "entry_pos":  entry_pos,
                "entry_z":    entry_z,
                "exit_z":     z.loc[t] if t in z.index else np.nan,
                "n_days":     (t - entry_date).days,
                "trade_pnl":  trade_pnl,
            })
            active    = False
            trade_pnl = 0.0

        if opened or flipped:
            active     = True
            entry_date = t
            entry_z    = z.loc[t] if t in z.index else np.nan
            entry_pos  = float(p)
            direction  = "LONG" if p > 0 else "SHORT"
            trade_pnl  = pnl.loc[t]
        elif active:
            trade_pnl += pnl.loc[t]

        prev_p = p

    if active:
        last_t = pos.index[-1]
        trade_rows_coint.append({
            "pair":       r.pair,
            "entry_date": entry_date,
            "exit_date":  last_t,
            "direction":  direction,
            "entry_pos":  entry_pos,
            "entry_z":    entry_z,
            "exit_z":     z.loc[last_t] if last_t in z.index else np.nan,
            "n_days":     (last_t - entry_date).days,
            "trade_pnl":  trade_pnl,
        })

trades_coint_df = pd.DataFrame(trade_rows_coint)
trades_coint_df.to_csv(OUT_DIR / "trades_table_coint.csv", index=False)
print("[OK] trades_table_coint.csv saved:", trades_coint_df.shape)

# --- Sanity check (same logic as cell-35) ---
extracted_counts = trades_coint_df.groupby("pair").size().rename("extracted")
flip_counts = {}
for r in results_coint:
    p = r.pos.values
    flips = ((np.sign(p[1:]) != np.sign(p[:-1])) &
             (p[1:] != 0) & (p[:-1] != 0)).sum()
    flip_counts[r.pair] = int(flips)
flip_counts = pd.Series(flip_counts, name="flips")

check_coint = pd.DataFrame({
    "Trades_bt":   bt_summary_coint.set_index("pair")["Trades"],
    "extracted":   extracted_counts,
    "flips":       flip_counts,
})
check_coint["expected_2x_extracted"] = check_coint["Trades_bt"] + check_coint["flips"]
check_coint["actual_2x_extracted"]   = 2 * check_coint["extracted"]
check_coint["mismatch"] = check_coint["expected_2x_extracted"] - check_coint["actual_2x_extracted"]

bad_coint = check_coint[check_coint["mismatch"].abs() > 1]
if len(bad_coint) > 0:
    print("\n[WARN] Cointegration-only baseline: trade-count sanity check failed:")
    print(bad_coint)
else:
    print(f"\n[OK] Baseline trade-count sanity check passed for all {len(check_coint)} pairs.")
    print(f"     Total extracted trades: {check_coint['extracted'].sum()}")
    print(f"     Total direct flips:     {check_coint['flips'].sum()}")

print("\n[NB2 v1.1 — COINTEGRATION-ONLY BASELINE EXPORTED]")
print("Additional artifacts for NB04 4-variant ablation:")
print(" • pairs_metadata_coint.csv")
print(" • pair_timeseries_coint.parquet")
print(" • trades_table_coint.csv")

## Key Takeaways

This notebook implements a **market-neutral OU-based pairs trading
framework** for Latin American ADRs under a strict in-sample/out-of-sample
regime, producing a frozen statistical baseline for downstream conditioning
layers.

**Methodology.** Pairs are selected on a four-stage pipeline: returns-based
correlation prefilter (|ρ| ≥ 0.35 on IS), Engle-Granger and ADF cointegration
tests (computed for transparency, not used as binding filters), AR(1) OU
estimation with viability bounds (φ ∈ (0,1), HL ∈ [5, 60] days, R² ≥ 0.05),
and a market-neutrality constraint (β > 0). Top-20 pairs by fastest
half-life are taken into the OOS backtest; hedge ratios (α, β) are frozen
on IS and applied unchanged in OOS. Trading is rule-based: enter at
|z| ≥ 2.0, exit at |z| < 0.5, with direct +1 ↔ −1 flips allowed when z
crosses both thresholds in a single bar.


**Rolling Sharpe shows regime decay.** All three portfolio variants
show 1Y rolling Sharpes peaking in late 2023 and declining steadily
into 2025, consistent with the structural regime shift documented in
the literature on pairs-trading research. This is the strongest signal in the
notebook that the IS-fitted parameter structure is not stable through
the OOS window — and is the primary motivation for the regime-aware
conditioning layers in downstream work.

**Position in the pipeline.** Results are reported in spread-PnL units
for relative comparison rather than capital-scaled deployment. This
notebook is a statistical baseline, not a deployable strategy. The
findings here — return concentration in a single pair, net-drag pairs,
regime decay through OOS, effective dimensionality below nominal —
motivate the downstream extensions in NB03: news/sentiment-based trade
conditioning, regime classification on macro data, dynamic sizing 
against rolling-vol estimates, and exclusion rules for net-drag pairs.

Portfolio construction comparisons and robustness diagnostics (equal-weight, inverse-vol, vol-target overlay, rolling Sharpe, effective-N, turnover) are explored in [`02b_ou_portfolio_appendix.ipynb`](appendix/02b_ou_portfolio_appendix.ipynb).

## NB03 Export Cell

In [ ]:
# ==============================================================================
# NB2 → NB3 EXPORT | STATISTICAL BASELINE HANDOFF
# ------------------------------------------------------------------------------
# Purpose:
# Persist the frozen OU-based statistical baseline produced in NB2 for
# downstream AI-conditioned diagnostics and trade filtering in NB3.
#
# Design principles:
# • Parameters and signals are fixed ex-ante (no look-ahead)
# • Outputs are model-agnostic and analytics-friendly
# • Artifacts support both time-series and trade-level analysis
# ==============================================================================

import json
import subprocess
from pathlib import Path

# ------------------------------------------------------------------------------
# 0. Output directory (versioned, notebook-scoped)
# ------------------------------------------------------------------------------
OUT_DIR = ARTIFACTS
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[OK] NB2 artifacts will be written to: {OUT_DIR.resolve()}")

# ------------------------------------------------------------------------------
# 1. Pair metadata (static / structural)
#    Used to define the invariant statistical identity of each pair
# ------------------------------------------------------------------------------
pair_meta = pairs_top[
    ["pair", "Ticker1", "Ticker2", "rho", "alpha", "beta",
     "phi", "halflife", "r2", "eg_p", "adf_p", "Nobs"]
].copy()

pair_meta.to_csv(OUT_DIR / "pairs_metadata.csv", index=False)

print("[OK] pairs_metadata.csv saved:", pair_meta.shape)

# ------------------------------------------------------------------------------
# 2. Pair-level time series (LONG format)
#    Granularity: (date × pair)
#    Used for regime tagging, AI annotation, and conditional diagnostics
#
#    Schema:
#    date | pair | spread | zscore | position | pnl
# ------------------------------------------------------------------------------
rows = []
oos_idx = price_test.index   # every pair is exported on the full OOS index
for r in results:
    df = pd.DataFrame({
        "date":     oos_idx,
        "pair":     r.pair,
        "spread":   S_pairs[r.pair].reindex(oos_idx),
        "zscore":   r.z.reindex(oos_idx),
        "position": r.pos.reindex(oos_idx),
        "pnl":      r.pnl.reindex(oos_idx).fillna(0.0),
    })
    rows.append(df)

pair_ts = (
    pd.concat(rows, axis=0)
      .sort_values(["pair", "date"])
      .reset_index(drop=True)
)

# Sanity: every pair should have full OOS coverage
expected_rows = len(price_test) * len(pairs_top)
assert len(pair_ts) == expected_rows, (
    f"pair_ts row count mismatch: got {len(pair_ts)}, "
    f"expected {expected_rows} = {len(price_test)} OOS days × {len(pairs_top)} pairs"
)

pair_ts.to_parquet(OUT_DIR / "pair_timeseries.parquet", index=False)

print("[OK] pair_timeseries.parquet saved:", pair_ts.shape)

# ------------------------------------------------------------------------------
# 3. Trade-level table (event-based representation)
#    Primary input for AI-conditioned trade diagnostics in NB3
#
#    Each row corresponds to a completed trade (direct-flip aware):
#    pair | entry_date | exit_date | direction | entry_pos |
#    entry_z | exit_z | n_days | trade_pnl
# ------------------------------------------------------------------------------
trade_rows = []

for r in results:
    pos = r.pos
    z   = r.z
    pnl = r.pnl
    s   = S_pairs[r.pair]

    active     = False
    entry_date = None
    entry_z    = None
    entry_pos  = None
    direction  = None
    trade_pnl  = 0.0
    prev_p     = 0.0

    for t in pos.index:
        p = pos.loc[t]

        opened  = (prev_p == 0) and (p != 0)
        closed  = (prev_p != 0) and (p == 0)
        flipped = (prev_p != 0) and (p != 0) and (np.sign(p) != np.sign(prev_p))

        # Close current trade on flat or direct flip
        if active and (closed or flipped):
            trade_rows.append({
                "pair":       r.pair,
                "entry_date": entry_date,
                "exit_date":  t,
                "direction":  direction,
                "entry_pos":  entry_pos,
                "entry_z":    entry_z,
                "exit_z":     z.loc[t] if t in z.index else np.nan,
                "n_days":     (t - entry_date).days,
                "trade_pnl":  trade_pnl,
            })
            active    = False
            trade_pnl = 0.0

        # Open new trade on flat-to-position or direct flip
        if opened or flipped:
            active     = True
            entry_date = t
            entry_z    = z.loc[t] if t in z.index else np.nan
            entry_pos  = float(p)
            direction  = "LONG" if p > 0 else "SHORT"
            trade_pnl  = pnl.loc[t]
        elif active:
            trade_pnl += pnl.loc[t]

        prev_p = p

    # Close any trade still open at end of OOS window
    if active:
        last_t = pos.index[-1]
        trade_rows.append({
            "pair":       r.pair,
            "entry_date": entry_date,
            "exit_date":  last_t,
            "direction":  direction,
            "entry_pos":  entry_pos,
            "entry_z":    entry_z,
            "exit_z":     z.loc[last_t] if last_t in z.index else np.nan,
            "n_days":     (last_t - entry_date).days,
            "trade_pnl":  trade_pnl,
        })

trades_df = pd.DataFrame(trade_rows)
trades_df.to_csv(OUT_DIR / "trades_table.csv", index=False)

print("[OK] trades_table.csv saved:", trades_df.shape)

# Sanity: extracted trades should match bt_summary trade counts
# bt_summary["Trades"] = count of position-change events (n_transitions)
# trades_df rows       = count of completed trades (n_round_trips + n_direct_flips)
# Relationship: 2 * n_extracted ≈ n_transitions + n_direct_flips
# For each pair, count direct flips (pos sign changes without crossing 0)

extracted_counts = trades_df.groupby("pair").size().rename("extracted")

flip_counts = {}
for r in results:
    p = r.pos.values
    flips = ((np.sign(p[1:]) != np.sign(p[:-1])) &
             (p[1:] != 0) & (p[:-1] != 0)).sum()
    flip_counts[r.pair] = int(flips)
flip_counts = pd.Series(flip_counts, name="flips")

check = pd.DataFrame({
    "Trades_bt":   bt_summary.set_index("pair")["Trades"],
    "extracted":   extracted_counts,
    "flips":       flip_counts,
})
check["expected_2x_extracted"] = check["Trades_bt"] + check["flips"]
check["actual_2x_extracted"]   = 2 * check["extracted"]
check["mismatch"] = check["expected_2x_extracted"] - check["actual_2x_extracted"]

bad = check[check["mismatch"].abs() > 1]   # allow ±1 for boundary effects
if len(bad) > 0:
    print("[WARN] Trade-count sanity check failed for pairs:")
    print(bad)
else:
    print(f"[OK] Trade-count sanity check passed for all {len(check)} pairs.")
    print(f"     Total extracted trades: {check['extracted'].sum()}")
    print(f"     Total direct flips:     {check['flips'].sum()}")

# ------------------------------------------------------------------------------
# 4. Run metadata (for traceability / version control)
# ------------------------------------------------------------------------------
try:
    git_hash = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=ROOT,
        stderr=subprocess.DEVNULL
    ).decode().strip()
except Exception:
    git_hash = "unknown"

run_meta = {
    "split_date":    str(split_date.date()),
    "n_train":       int(len(price_train)),
    "n_test":        int(len(price_test)),
    "n_pairs":       int(len(pairs_top)),
    "config":        CONFIG,
    "git_hash":      git_hash,
    "universe_size": int(len(universe)),
}

with open(OUT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_meta, f, indent=2, default=str)

print("[OK] run_metadata.json saved.")

# ------------------------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------------------------
print("\n[NB2 LOCKED — STATISTICAL BASELINE FINALIZED]")
print("Exported artifacts for NB3 (AI-Conditioned Pairs Engine):")
print(" • pairs_metadata.csv       (structural parameters — 12 cols)")
print(" • pair_timeseries.parquet  (date × pair signals)")
print(" • trades_table.csv         (event-level trade outcomes — 9 cols)")
print(" • run_metadata.json        (run config + git hash)")

## References

- Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). "Pairs Trading: Performance of a Relative-Value Arbitrage Rule." *The Review of Financial Studies*, 19(3), 797–827.
- Do, B., & Faff, R. (2010). "Does Simple Pairs Trading Still Work?" *Financial Analysts Journal*, 66(4), 83–95.
- Elliott, R. J., van der Hoek, J., & Malcolm, W. P. (2005). "Pairs Trading." *Quantitative Finance*, 5(3), 271–276.